# Deploy MCP Server to Amazon Bedrock AgentCore Runtime

This notebook deploys the `mcp_server.py` to AgentCore Runtime using AWS IAM inbound authentication.

## Install Dependencies

In [7]:
#!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup

In [1]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

tool_name = "blog_mcp_math"

print(f"Using AWS region: {region}")

/Users/mingyyzz/dev-local/aws.blog1/.venv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Using AWS region: us-west-2


## Configure and Launch AgentCore Runtime

In [4]:
required_files = ["mcp_server.py"]
agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed")

print("Launching MCP server to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()
print("Launch completed")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

Entrypoint parsed: file=/Users/mingyyzz/dev-local/aws.blog1/src/mcp_server.py, bedrock_agentcore_name=mcp_server
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: blog_mcp_math
Memory disabled
Network mode: PUBLIC


Configuring AgentCore Runtime...


📄 Using existing Dockerfile: /Users/mingyyzz/dev-local/aws.blog1/src/Dockerfile

Generated .dockerignore: /Users/mingyyzz/dev-local/aws.blog1/src/.dockerignore
Keeping 'blog_mcp_math' as default agent
Bedrock AgentCore configured: /Users/mingyyzz/dev-local/aws.blog1/src/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'blog_mcp_math' to account 980413094772 (us-west-2)
Generated image tag: 20260322-043155-713
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: blog_mcp_math


Configuration completed
Launching MCP server to AgentCore Runtime...


ECR repository available: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_mcp_math
Getting or creating execution role for agent: blog_mcp_math
Using AWS region: us-west-2, account ID: 980413094772
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-622e8ad068


✅ Reusing existing ECR repository: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_mcp_math


✅ Reusing existing execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-622e8ad068
Execution role available: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-622e8ad068
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: blog_mcp_math
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-622e8ad068
Reusing existing CodeBuild execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-622e8ad068
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: blog_mcp_math/source.zip
Updated CodeBuild project: bedrock-agentcore-blog_mcp_math-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.4s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE

Launch completed
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:980413094772:runtime/blog_mcp_math-Z057Lr3Pdg
Agent ID: blog_mcp_math-Z057Lr3Pdg


## Store Agent ARN in Parameter Store

In [14]:
ssm_client.put_parameter(
    Name="/blog_mcp_math/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_mcp_math MCP server",
    Overwrite=True,
)
print("Agent ARN stored in Parameter Store")
print(f"Agent ARN: {launch_result.agent_arn}")

Agent ARN stored in Parameter Store
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:980413094772:runtime/blog_mcp_math-Z057Lr3Pdg


## Test the MCP

In [ ]:
!cd test && pytest mcp_test.py -v

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_mcp_math/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True,
# )